# 第 2 课：从零实现最小 Agent Loop

预计用时：60–90 分钟  
适合人群：完成上一课的零基础学习者；本 Notebook 也包含独立运行所需的准备代码。

## 学习目标

- 理解 Agent Loop 的四个阶段
- 看懂模型消息与 tool call 的往返
- 用超时、步数和重复检测阻止失控循环

## 学习方式

按顺序运行每个代码单元格。先阅读预测结果，再运行验证；遇到报错先看本课“常见问题”，不要直接跳过。带有真实模型或外网请求的示例默认注释，确认 API Key 与费用后再启用。


## 1. 先理解概念

Agent 不是一次模型调用，而是一个受控循环：模型决定是否调用工具，程序执行工具，把结果作为 `tool` 消息送回模型，直到模型给出最终回答。工程可靠性来自循环边界与错误处理，而不是提示词本身。

### 本课路线

1. 创建工具注册表
2. 把 Pydantic 模型转换为 OpenAI 工具 Schema
3. 发送消息给模型
4. 执行并回传工具结果
5. 在达到终止条件时返回答案


## 2. 运行前检查

1. 从项目根目录启动 Jupyter Lab。
2. 选择项目 `.venv` 对应的 Python 内核。
3. 若本课调用百炼，先在启动 Jupyter 的终端设置 `DASHSCOPE_API_KEY`。
4. 不要把 Key 粘贴到单元格、截图或 Git 提交中。

> 下方“准备代码”可能与前课重复，这是为了保证每个 Notebook 都能单独运行。初学时建议展开阅读，熟悉后可折叠。


### 准备代码


In [ ]:
# %pip install -q openai pydantic>=2.7 httpx fastapi uvicorn fastmcp langgraph langfuse ragas numpy pytest

import os
from dotenv import load_dotenv

load_dotenv()

# 推荐在启动 Jupyter 前设置：
# Windows PowerShell: $env:DASHSCOPE_API_KEY='sk-...'
# macOS/Linux:       export DASHSCOPE_API_KEY='sk-...'

BAILIAN_API_KEY = os.getenv('DASHSCOPE_API_KEY', '')
BAILIAN_BASE_URL = os.getenv(
    'BAILIAN_BASE_URL',
    'https://dashscope.aliyuncs.com/compatible-mode/v1',
)
BAILIAN_MODEL = os.getenv('BAILIAN_MODEL', 'qwen-plus')
BAILIAN_EMBEDDING_MODEL = os.getenv('BAILIAN_EMBEDDING_MODEL', 'text-embedding-v4')

print('模型:', BAILIAN_MODEL)
print('Base URL:', BAILIAN_BASE_URL)
print('API Key:', '已配置' if BAILIAN_API_KEY else '未配置（调用模型前必须设置）')


### 准备代码


In [ ]:
from __future__ import annotations

import asyncio
import json
import logging
import math
import sqlite3
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Awaitable, Callable, Literal, TypedDict

import httpx
import numpy as np
from openai import AsyncOpenAI
from pydantic import BaseModel, ConfigDict, Field, ValidationError

WORKSPACE = (Path.cwd() / 'agent_workspace').resolve()
WORKSPACE.mkdir(exist_ok=True)

def require_api_key() -> None:
    if not BAILIAN_API_KEY:
        raise RuntimeError('请先设置环境变量 DASHSCOPE_API_KEY，然后重新运行配置单元格。')

client = AsyncOpenAI(api_key=BAILIAN_API_KEY or 'missing', base_url=BAILIAN_BASE_URL)
print('工作目录:', WORKSPACE)


### 准备代码


In [ ]:
class AgentLimits(BaseModel):
    model_config = ConfigDict(extra='forbid')
    max_steps: int = Field(default=8, ge=1, le=30)
    model_timeout_s: float = Field(default=45, gt=0, le=300)
    tool_timeout_s: float = Field(default=15, gt=0, le=120)
    total_timeout_s: float = Field(default=120, gt=0, le=600)

class ToolResult(BaseModel):
    ok: bool
    data: Any = None
    error: str | None = None
    retryable: bool = False

ToolHandler = Callable[[BaseModel], Awaitable[Any]]

@dataclass
class RegisteredTool:
    name: str
    description: str
    args_model: type[BaseModel]
    handler: ToolHandler
    side_effect: bool = False

    def openai_schema(self) -> dict[str, Any]:
        schema = self.args_model.model_json_schema()
        schema['additionalProperties'] = False
        return {
            'type': 'function',
            'function': {
                'name': self.name,
                'description': self.description,
                'parameters': schema,
            },
        }

class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, RegisteredTool] = {}

    def register(self, tool: RegisteredTool) -> None:
        if tool.name in self._tools:
            raise ValueError(f'工具重复注册: {tool.name}')
        self._tools[tool.name] = tool

    @property
    def schemas(self) -> list[dict[str, Any]]:
        return [tool.openai_schema() for tool in self._tools.values()]

    async def execute(self, name: str, raw_arguments: str, timeout_s: float) -> ToolResult:
        tool = self._tools.get(name)
        if tool is None:
            return ToolResult(ok=False, error=f'未知工具: {name}', retryable=False)
        try:
            arguments = json.loads(raw_arguments or '{}')
            validated = tool.args_model.model_validate(arguments)
        except json.JSONDecodeError as exc:
            return ToolResult(ok=False, error=f'工具参数不是合法 JSON: {exc}')
        except ValidationError as exc:
            return ToolResult(ok=False, error=f'工具参数校验失败: {exc}')
        try:
            async with asyncio.timeout(timeout_s):
                value = await tool.handler(validated)
            return ToolResult(ok=True, data=value)
        except TimeoutError:
            return ToolResult(ok=False, error=f'工具 {name} 执行超时', retryable=True)
        except httpx.HTTPStatusError as exc:
            retryable = exc.response.status_code in {408, 429, 500, 502, 503, 504}
            return ToolResult(ok=False, error=f'上游 HTTP {exc.response.status_code}', retryable=retryable)
        except Exception as exc:
            return ToolResult(ok=False, error=f'{type(exc).__name__}: {exc}', retryable=False)


### 核心实验


In [ ]:
class MinimalAgent:
    def __init__(self, registry: ToolRegistry, limits: AgentLimits | None = None) -> None:
        self.registry = registry
        self.limits = limits or AgentLimits()

    async def run(self, user_input: str) -> str:
        require_api_key()
        messages: list[dict[str, Any]] = [
            {'role': 'system', 'content': '你是可靠的中文助手。需要外部事实或操作时调用工具；工具失败时不得编造结果。'},
            {'role': 'user', 'content': user_input},
        ]
        repeated_calls: dict[str, int] = {}

        async with asyncio.timeout(self.limits.total_timeout_s):
            for step in range(1, self.limits.max_steps + 1):
                async with asyncio.timeout(self.limits.model_timeout_s):
                    response = await client.chat.completions.create(
                        model=BAILIAN_MODEL,
                        messages=messages,
                        tools=self.registry.schemas or None,
                        tool_choice='auto' if self.registry.schemas else None,
                        temperature=0.2,
                    )
                message = response.choices[0].message
                assistant_message: dict[str, Any] = {
                    'role': 'assistant',
                    'content': message.content or '',
                }
                if message.tool_calls:
                    assistant_message['tool_calls'] = [tc.model_dump() for tc in message.tool_calls]
                messages.append(assistant_message)

                if not message.tool_calls:
                    return message.content or ''

                for call in message.tool_calls:
                    fingerprint = f'{call.function.name}:{call.function.arguments}'
                    repeated_calls[fingerprint] = repeated_calls.get(fingerprint, 0) + 1
                    if repeated_calls[fingerprint] > 2:
                        result = ToolResult(ok=False, error='相同工具调用重复过多，已阻止循环')
                    else:
                        result = await self.registry.execute(
                            call.function.name,
                            call.function.arguments,
                            self.limits.tool_timeout_s,
                        )
                    messages.append({
                        'role': 'tool',
                        'tool_call_id': call.id,
                        'content': result.model_dump_json(),
                    })

        raise RuntimeError(f'Agent 超过最大步骤数 {self.limits.max_steps}')


## 3. 观察与验证

核心代码中的真实 API 调用默认被注释。先运行无需额度的断言或定义单元格；确认输出和预期一致后，再逐行取消示例注释。


## 4. 代码讲解

重点跟踪 `messages` 如何增长。模型产生 `tool_calls` 时，必须把 assistant 消息和每个对应的 tool 消息都追加进去；没有工具调用时才结束。`fingerprint` 防止模型反复发出完全相同的请求。

调试建议：从报错的最后一行开始读，确认当前 Notebook 的单元格是否按顺序全部运行；若看到 `NameError`，通常是准备单元格未运行或内核已重启。


## 5. 常见问题

- **`ModuleNotFoundError`**：确认选中了 `.venv` 内核，并重新安装 `requirements.txt`。
- **提示未配置 API Key**：在启动 Jupyter 的同一个终端设置环境变量，然后重启内核。
- **网络超时或 429**：公开接口或模型服务可能限流；稍后重试，不要移除超时保护。
- **运行结果和预期不同**：先执行“Restart Kernel and Run All”，排除旧变量残留。
- **产生费用吗？**：只有实际调用百炼聊天或 Embedding 接口才会消耗额度；本地定义、SQLite 和断言不会。

## 6. 练习

- 画出一次含工具调用的消息时序图
- 把 `max_steps` 改成 1，观察边界行为
- 给 `MinimalAgent.run` 增加每一步的调试输出

建议先复制相关单元格再修改，保留一份能工作的基线。


## 7. 本课验收

完成后逐项确认：

- [ ] 能复述循环的四个阶段
- [ ] 知道 `tool_call_id` 为什么必须原样返回
- [ ] 能指出三种终止或失败路径

如果某项还解释不清，回到对应代码，用更小的输入单独调用函数，而不是直接运行完整 Agent。


## 下一步

继续学习 `03_Function_Calling工具.ipynb`。

> 学习记录建议：写下今天最重要的一个概念、遇到的一个错误、以及你如何验证修复。
